# HVAC Tag+Symbol Detector — YOLOv8s Training (Kaggle T4)

Trains a 2-class YOLOv8s on the auto-labeled HVAC crops dataset.
Classes: `0=symbol`, `1=tag_bubble`.

**Setup before Run All:**
1. `Settings` (right panel) → `Accelerator: GPU T4 x2` (or GPU P100)
2. `Settings` → `Add data` → Upload `yolo_tag_dataset.zip` as a new private dataset. Name it e.g. `hvac-tag-yolo-v1`.
3. Edit `KAGGLE_DATASET_SLUG` below to match the dataset slug Kaggle assigns.
4. Run All.

**Expected runtime:** 2-3 hours for 60 epochs on T4.
**Output:** `best.pt` + training plots in `/kaggle/working/runs/detect/train/weights/`.

In [ ]:
# 1) Verify GPU + install ultralytics
!nvidia-smi
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 2) Locate the uploaded dataset and unzip into /kaggle/working
# Adjust this slug to match your uploaded Kaggle dataset name
KAGGLE_DATASET_SLUG = 'hvac-tag-yolo-v1'

import os, glob, time
input_dir = f'/kaggle/input/{KAGGLE_DATASET_SLUG}'
print('Contents of input dir:')
!ls -la {input_dir}

# Find the zip (Kaggle sometimes nests it)
zips = glob.glob(f'{input_dir}/**/yolo_tag_dataset.zip', recursive=True) + \
       glob.glob(f'{input_dir}/yolo_tag_dataset.zip')
assert zips, f'Could not find yolo_tag_dataset.zip under {input_dir}'
zip_path = zips[0]
print(f'Zip: {zip_path}')

os.makedirs('/kaggle/working/yolo_tag_dataset', exist_ok=True)
t0 = time.time()
!unzip -q -o {zip_path} -d /kaggle/working/yolo_tag_dataset
print(f'Unzipped in {time.time()-t0:.1f}s')
!ls /kaggle/working/yolo_tag_dataset
!find /kaggle/working/yolo_tag_dataset/images -name '*.png' | wc -l

In [ ]:
# 3) Regenerate data.yaml with Kaggle paths (the one in the zip has local paths)
DATA_ROOT = '/kaggle/working/yolo_tag_dataset'
with open(f'{DATA_ROOT}/data.yaml', 'w') as f:
    f.write(f"""path: {DATA_ROOT}
train: images/train
val: images/val

nc: 2
names:
  0: symbol
  1: tag_bubble
""")
!cat {DATA_ROOT}/data.yaml

n_train = len([p for p in os.listdir(f'{DATA_ROOT}/images/train')])
n_val = len([p for p in os.listdir(f'{DATA_ROOT}/images/val')])
print(f'\nTrain: {n_train}   Val: {n_val}')

In [ ]:
# 4) Train YOLOv8s
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=f'{DATA_ROOT}/data.yaml',
    epochs=60,
    imgsz=320,
    batch=64,
    device=0,
    project='/kaggle/working/runs/detect',
    name='train',
    patience=15,
    cache='ram',
    amp=True,
    lr0=0.01,
    warmup_epochs=3,
    close_mosaic=10,
    save_period=10,
)

In [ ]:
# 5) Validation metrics + show confusion matrix / PR curve
from ultralytics import YOLO
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
metrics = model.val(data=f'{DATA_ROOT}/data.yaml', imgsz=320, batch=64, device=0)
print()
print('Per-class mAP50-95:')
for i, name in enumerate(['symbol', 'tag_bubble']):
    print(f'  {name:12s}  mAP50={metrics.box.ap50[i]:.3f}  mAP50-95={metrics.box.ap[i]:.3f}')

In [ ]:
# 6) Quick visual check on 12 val images
import glob, random
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
val_imgs = glob.glob(f'{DATA_ROOT}/images/val/*.png')
random.seed(7)
samples = random.sample(val_imgs, min(12, len(val_imgs)))

out_dir = '/kaggle/working/val_preview'
os.makedirs(out_dir, exist_ok=True)
results = model.predict(samples, imgsz=320, conf=0.25, save=True,
                        project=out_dir, name='preview', exist_ok=True)
for p in sorted(glob.glob(f'{out_dir}/preview/*.png'))[:12]:
    display(IPImage(p))

In [ ]:
# 7) Copy best.pt to a known name for easy download
import shutil
shutil.copy('/kaggle/working/runs/detect/train/weights/best.pt',
            '/kaggle/working/hvac_tag_detector_v1.pt')
shutil.copy('/kaggle/working/runs/detect/train/results.png',
            '/kaggle/working/training_results.png')
print('Final artifacts in /kaggle/working/:')
!ls -la /kaggle/working/*.pt /kaggle/working/*.png
print()
print('Download hvac_tag_detector_v1.pt from the right panel (Output section).')